In [0]:
# 99_validate_run – row counts + duplicate keys per layer, appended to ops.run_validation
from datetime import datetime, timezone

dbutils.widgets.text("label", "")
dbutils.widgets.text("adf_run_id", "")
LABEL      = dbutils.widgets.get("label").strip() or "manual"
# LABEL = "after_run2"
ADF_RUN_ID = dbutils.widgets.get("adf_run_id").strip() or None
CAT        = "laddstolpar_df"
CHECK_TS   = datetime.now(timezone.utc)

# table -> grain key (bronze ledger tables: key + _file = "same file read twice")
TABLES = {
    "bronze.elpris":             ["elomrade", "time_start"],
    "bronze.scb_tab3277":        ["region", "drivmedel", "contentscode", "tid", "_file"],
    "bronze.scb_tab628":         ["region", "kon", "contentscode", "tid", "_file"],
    "bronze.scb_tab3276":        ["region", "agarkategori", "contentscode", "tid", "_file"],
    "bronze.trafa_t10026":       ["ar", "reglan", "regkom", "drivmedel", "_file"],
    "bronze.nobil_stations":     ["station_id", "_snapshot_date"],
    "bronze.nobil_stats_county": ["countyid", "_snapshot_date"],
    "silver.elpris":             ["elomrade", "time_start_utc"],
    "silver.scb_tab3277":        ["kommun_kod", "drivmedel_scb_kod", "contentscode", "period"],
    "silver.scb_tab628":         ["kommun_kod", "contentscode", "ar"],
    "silver.scb_tab3276":        ["kommun_kod", "agarkategori", "contentscode", "ar"],
    "silver.trafa_t10026":       ["kommun_kod", "drivmedel_kod", "ar"],
    "silver.nobil_station":      ["station_id"],
    "silver.nobil_connector":    ["station_id", "connector_no"],
    "gold.dim_kommun":               ["kommun_kod"],
    "gold.fct_nyregistrering_manad": ["kommun_kod", "scb_period", "drivmedel"],
    "gold.fct_fordon_i_trafik_ar":   ["kommun_kod", "ar", "drivmedel"],
    "gold.fct_elpris_dag":           ["elomrade", "datum"],
    "gold.fct_kommun_ar":            ["kommun_kod", "ar"],
    "gold.fct_laddstation":          ["station_id"],
    "gold.fct_trafik_kommun":        ["kommun_kod"],
    "gold.mart_allocation_combined": ["n_p1", "variant", "kommun_kod"],
}

rows = []
for name, key in TABLES.items():
    fq   = f"{CAT}.{name}"
    cur  = " WHERE __END_AT IS NULL" if "__END_AT" in spark.table(fq).columns else ""
    k    = ", ".join(f"`{c}`" for c in key)
    n_rows    = spark.sql(f"SELECT count(*) AS n FROM {fq}").first().n
    n_current = spark.sql(f"SELECT count(*) AS n FROM {fq}{cur}").first().n
    n_dup     = spark.sql(f"""SELECT coalesce(sum(n - 1), 0) AS d FROM (
                                SELECT count(*) AS n FROM {fq}{cur} GROUP BY {k} HAVING count(*) > 1)""").first().d
    ver = op = op_ts = None
    if name.startswith("bronze."):
        h = spark.sql(f"DESCRIBE HISTORY {fq} LIMIT 1").first()
        ver, op, op_ts = h.version, h.operation, h.timestamp
    rows.append((CHECK_TS, LABEL, ADF_RUN_ID, name.split(".")[0], name,
                 n_rows, n_current, int(n_dup), ver, op, op_ts))

schema = """check_ts timestamp, label string, adf_run_id string, layer string, table_name string,
            n_rows bigint, n_current bigint, n_dup_key bigint,
            last_version bigint, last_operation string, last_operation_ts timestamp"""
df = spark.createDataFrame(rows, schema)
df.write.mode("append").saveAsTable(f"{CAT}.ops.run_validation")
display(df.orderBy("layer", "table_name"))

# Fail loudly (→ ADF failure mail when this runs as a pipeline step)
dups = df.filter("n_dup_key > 0").select("table_name").collect()
assert not dups, f"Duplicate keys in: {[r.table_name for r in dups]}"